# Corte 3, Punto 2: flujo continuo y ventanas temporales

Por: Gabriel Rodriguez y Jonatan Palomares

**Subtareas:** T2.1 a T2.6 (simulación, micro-lotes, lectura en streaming, ventanas, llegada tardía y validación contra el cálculo por lotes).

Entrada: gold.

_Por hacer._

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [1]:
# Importación de librerías

import os
import time
from time import perf_counter
from statistics import median
import random
import math
import json
import urllib.request
from IPython.display import display, Markdown

import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.types import DateType, TimestampType
from pyspark import StorageLevel

from pamd.config import (
    INPUT_CSV,
    BRONZE_PARQUET_UNPARTITIONED,
    BRONZE_PARQUET,
    QUARANTINE_PARQUET,
    SILVER_PARQUET,
    GOLD_PARQUET,
    STREAMING_LOTES,
    STREAMING_ENTRADA,
    STREAMING_TARDIOS,
    STREAMING_CHECKPOINT,
    WRITE_MODE,
    OUTPUT_PARTITIONS,
)
from pamd.spark_session import get_spark
from pamd.calidad import marcar_atipicos, unificar_nit_entidad
from pamd.flujo import (
    base_del_flujo, plan_de_lotes, asignar_lotes, huella_por_lote, entregar_lote, vaciar, crear_carpeta,
)

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()


## T2.1 Diseño de la simulación

Aprobado por Gabriel Rodríguez el 15-nov-2026 (criterio 3 de T2.1); se informa a Jonatan Palomares. Todos los parámetros están en la celda siguiente y se pueden cambiar sin tocar el resto del notebook.

**Qué se simula.** SECOP II no publica los contratos como un flujo continuo, así que el flujo se simula: gold se parte en archivos (micro-lotes) que llegan uno a uno a una carpeta de MinIO, y Spark Structured Streaming los procesa a medida que aparecen.

| Aspecto | Decisión |
|---|---|
| Origen | gold con las marcas de `src/pamd/calidad.py` (`marcar_atipicos(unificar_nit_entidad(gold), silver)`), solo con las columnas que usa el flujo |
| Lotes | `NUM_LOTES` = 20 lotes regulares en orden de `fecha_adjudicacion`, cortados por meses completos (un mes nunca queda partido entre dos lotes) y con un número parecido de contratos |
| Carpetas | los lotes se preparan en `streaming/lotes/` y se copian uno a uno a `streaming/entrada/`, que es la carpeta que lee el flujo |
| Ritmo | llega un lote cada `PAUSA_SEGUNDOS` = 5 s; el flujo lee con `maxFilesPerTrigger = 1`, un lote por disparo |
| Qué es tardío | un contrato que llega en un lote posterior a otro que ya traía meses más recientes que el suyo |
| Tardíos inyectados | dos lotes extra con el `PORCENTAJE_TARDIO` = 20 % de los contratos de un mes, elegidos con un hash de `id_adjudicacion` y la `SEMILLA` = 13 (misma semilla, mismos lotes). **Dentro de la tolerancia:** penúltimo mes del lote 10, llega justo después del lote 10. **Fuera de la tolerancia:** primer mes del lote 5, llega después del lote 15 |
| Ventana | mes calendario exacto: se agrupa por el inicio del mes (`date_trunc`) y por el departamento de la entidad, con `withWatermark` sobre ese inicio de mes. La versión anual se obtiene sumando los meses de cada año |
| Tolerancia | `TOLERANCIA` = 75 días antes del mes más reciente visto: se aceptan los dos meses anteriores. 75 días nunca cae en un día 1, así que no hay casos en el límite |
| Fuera de la tolerancia | Spark descarta el registro en la agregación; el flujo lo guarda en `streaming/tardios/` y su ventana se corrige sumando esa tabla, así que ninguno se pierde en silencio |
| Medidas | las mismas de las tablas de T1.4: contratos, contratos atípicos, contratos con valor cero, `valor_total` (sin atípicos) y `valor_atipicos` |
| Salida | modo `update` con `foreachBatch`: en cada disparo llegan solo las ventanas que cambiaron. En T2 se guardan en una tabla en memoria; el destino idempotente en MinIO es la T3.2 |
| Punto de control | `streaming/checkpoint/ventanas` (T3.1) |

**Por qué mes calendario y no `window()` de duración fija:** el mes exacto coincide con `anio_mes` de las tablas resumen y del dashboard, y como el inicio del mes es una marca de tiempo sigue admitiendo marca de agua; una ventana fija de 30 días no coincide con los meses.

In [2]:
# T2.1 · Parámetros de la simulación (se cambian aquí; el resto del notebook los usa)
NUM_LOTES = 20                         # lotes regulares, en orden de fecha_adjudicacion y por meses completos
PAUSA_SEGUNDOS = 5                     # espera entre la llegada de un lote y el siguiente
SEMILLA = 13                           # semilla del hash que elige los contratos tardíos
PORCENTAJE_TARDIO = 20                 # % de contratos del mes elegido que se retienen y llegan tarde
TOLERANCIA = "75 days"                 # marca de agua: se aceptan los dos meses anteriores al más reciente visto
DISPARO = "2 seconds"                  # cada cuánto revisa el flujo si llegó un archivo nuevo
PARTICIONES_ESTADO = 4                 # particiones del estado de la agregación (quedan fijadas en el checkpoint)
SIN_DEPARTAMENTO = "SIN DEPARTAMENTO"  # misma categoría que las tablas de T1.4

# Lotes tardíos: lote regular del que sale el mes, qué mes de ese lote (0 = el primero, -2 = el penúltimo) y tras qué lote llega
TARDIOS = {
    "tardio_dentro": {"lote_origen": 10, "mes_en_lote": -2, "llega_tras_lote": 10},  # dentro de la tolerancia
    "tardio_fuera": {"lote_origen": 5, "mes_en_lote": 0, "llega_tras_lote": 15},     # fuera de la tolerancia
}

assert all(1 <= t["lote_origen"] <= t["llega_tras_lote"] <= NUM_LOTES for t in TARDIOS.values()), \
    "Un lote tardío debe llegar después de su lote de origen"
print(f"{NUM_LOTES} lotes regulares + {len(TARDIOS)} tardíos = {NUM_LOTES + len(TARDIOS)} archivos; "
      f"un lote cada {PAUSA_SEGUNDOS} s; tolerancia de {TOLERANCIA}")

20 lotes regulares + 2 tardíos = 22 archivos; un lote cada 5 s; tolerancia de 75 days


## T2.2 Generador de micro-lotes

Gold se parte en los lotes que fija T2.1 y cada lote se guarda como un único Parquet en `streaming/lotes/archivo=<nombre>/`. La entrega a la carpeta de entrada (copia de un lote, pausa, siguiente lote) la hace `entregar_lote` de `src/pamd/flujo.py` en T2.3, con el flujo ya encendido. El código del generador está en ese módulo porque el notebook 06 simula el mismo flujo.

In [3]:
# T2.2 · Plan de lotes: qué meses lleva cada archivo y en qué orden llega
gold = spark.read.parquet(GOLD_PARQUET)
silver = spark.read.parquet(SILVER_PARQUET)
n_gold = gold.count()

base = base_del_flujo(gold, silver).persist(StorageLevel.MEMORY_AND_DISK)
assert base.count() == n_gold, "Las marcas de calidad cambiaron el número de contratos"

orden = plan_de_lotes(base, NUM_LOTES, TARDIOS)
lotes_df = asignar_lotes(base, orden, SEMILLA, PORCENTAJE_TARDIO)
print(f"Gold: {n_gold:,} contratos -> {len(orden)} archivos")

Gold: 530,192 contratos -> 22 archivos


### Escritura de los lotes

In [4]:
# T2.2 · Escritura de los lotes (un Parquet por lote)
# True la primera vez (escribe los lotes en MinIO); False reutiliza los que ya están.
# Los lotes se escribieron el 15-nov-2026 con True; la salida de abajo es la de esa ejecución.
RECONSTRUIR_LOTES = False

if RECONSTRUIR_LOTES:
    (
        lotes_df
        .repartition("archivo")  # cada lote queda entero en una sola tarea, así que se escribe en un solo archivo
        .write.mode(WRITE_MODE).option("compression", "snappy")
        .partitionBy("archivo")
        .parquet(STREAMING_LOTES)
    )

lotes_escritos = spark.read.parquet(STREAMING_LOTES)

### Lista de lotes y criterios de aceptación

In [5]:
# T2.2 · Lista de lotes con su número de filas y la marca de tardío, y criterios de aceptación
huella_escrita = huella_por_lote(lotes_escritos)
archivos_por_lote = {
    r["archivo"]: r["count"]
    for r in lotes_escritos.select("archivo", F.input_file_name().alias("ruta")).distinct().groupBy("archivo").count().collect()
}

lista_lotes = pd.DataFrame([
    {
        "orden de llegada": i,
        "archivo": e["archivo"],
        "tardío": e["tipo"] != "regular",
        "meses": e["meses"][0] if len(e["meses"]) == 1 else f"{e['meses'][0]} a {e['meses'][-1]}",
        "filas": huella_escrita[e["archivo"]][0],
    }
    for i, e in enumerate(orden, 1)
])
display(lista_lotes.style.format({"filas": "{:,}"}))

# (1) La suma de filas de todos los lotes es igual a las filas de gold, y cada lote es un solo archivo
assert set(huella_escrita) == {e["archivo"] for e in orden}, "Los lotes escritos no coinciden con el plan"
assert lista_lotes["filas"].sum() == n_gold, "Los lotes no suman las filas de gold"
assert all(n == 1 for n in archivos_por_lote.values()), "Algún lote quedó partido en varios archivos"
assert lotes_escritos.select("id_del_proceso", "id_adjudicacion").distinct().count() == n_gold, "Un contrato quedó en dos lotes"

# (2) Con la misma semilla se generan los mismos lotes: se recalculan desde cero y se comparan con los escritos
huella_otra_vez = huella_por_lote(asignar_lotes(base, plan_de_lotes(base, NUM_LOTES, TARDIOS), SEMILLA, PORCENTAJE_TARDIO))
assert huella_otra_vez == huella_escrita, "La misma semilla no reproduce los mismos lotes"

# (3) Se sabe exactamente qué lotes son tardíos, de qué mes salen y cuándo llegan
for e in orden:
    if e["tipo"] != "regular":
        filas = huella_escrita[e["archivo"]][0]
        print(f"{e['archivo']}: {filas:,} contratos de {e['mes_retenido']} (del lote regular {e['lote_origen']}), "
              f"llegan después del lote regular {TARDIOS[e['tipo']]['llega_tras_lote']}")
n_tardios = int(lista_lotes.loc[lista_lotes["tardío"], "filas"].sum())
print(f"Tardíos: {n_tardios:,} contratos, el {100 * n_tardios / n_gold:.2f} % de gold")
print("Criterios de aceptación de T2.2: OK")

,orden de llegada,archivo,tardío,meses,filas
0,1,lote_01_regular,False,2015-06 a 2018-04,"27,889"
1,2,lote_02_regular,False,2018-05 a 2019-01,"25,620"
2,3,lote_03_regular,False,2019-02 a 2019-11,"28,617"
3,4,lote_04_regular,False,2019-12 a 2020-09,"27,177"
4,5,lote_05_regular,False,2020-10 a 2021-04,"23,720"
5,6,lote_06_regular,False,2021-05 a 2021-10,"30,431"
6,7,lote_07_regular,False,2021-11 a 2022-03,"26,812"
7,8,lote_08_regular,False,2022-04 a 2022-07,"21,842"
8,9,lote_09_regular,False,2022-08 a 2022-12,"34,509"
9,10,lote_10_regular,False,2023-01 a 2023-05,"20,887"


lote_11_tardio_dentro: 1,035 contratos de 2023-04 (del lote regular 10), llegan después del lote regular 10
lote_17_tardio_fuera: 866 contratos de 2020-10 (del lote regular 5), llegan después del lote regular 15
Tardíos: 1,901 contratos, el 0.36 % de gold
Criterios de aceptación de T2.2: OK
